# moviepy — Final composite

Composes the polls-tutorial demo into a single mp4 from:

- per-scene narration: `audio/scene_NN.mp3` (committed; rendered by `narrator.ipynb`)
- per-scene Manim clips: `media/videos/source/480p15/<SceneClass>.mp4` (gitignored; rendered by `manim.ipynb` — re-run that notebook to regenerate)

**Strategy** — build video and audio tracks **independently**, then attach at the very end:

1. Per scene, concatenate Manim sub-clips back-to-back. If the result is shorter than the narration, hold the last frame to fill. (Each scene has no audio at this stage.)
2. Concatenate all 9 silent scene clips into `final_video`.
3. Build the audio track as a `CompositeAudioClip` of the 9 narration `mp3`s placed at each scene's cumulative start time. Gaps between scenes (tail silence) become natural silence in the mix.
4. Apply a small gain boost (`AUDIO_GAIN`, default `1.5×` ≈ +3.5 dB) to compensate for moviepy's mono→stereo attenuation during AAC encode.
5. Attach the audio track to `final_video` and write `out/polls_demo.mp4` with a 192 kbps AAC track for clean speech reproduction.

`out/` is gitignored — re-run this notebook to regenerate on a fresh checkout.


In [ ]:
from pathlib import Path

import numpy as np

from moviepy import (
    AudioFileClip,
    CompositeAudioClip,
    ImageClip,
    VideoFileClip,
    concatenate_videoclips,
)

# Notebook lives in source/, so paths are relative to that.
AUDIO_DIR = Path("audio")
VIDEO_DIR = Path("media/videos/source/480p15")
OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "polls_demo.mp4"

assert AUDIO_DIR.exists(), f"missing {AUDIO_DIR.resolve()}"
assert VIDEO_DIR.exists(), (
    f"missing {VIDEO_DIR.resolve()} — re-run manim.ipynb to render the scene mp4s"
)

print(f"audio dir: {AUDIO_DIR.resolve()}")
print(f"video dir: {VIDEO_DIR.resolve()}")
print(f"out      : {OUT_PATH.resolve()}")


In [ ]:
# Per-scene assets. Order matches the conte.md storyboard.
SCENES = [
    {"id": 1, "audio": "scene_01.mp3", "videos": ["Scene1Opening.mp4"]},
    {"id": 2, "audio": "scene_02.mp3", "videos": ["Scene2Reinhardt.mp4"]},
    {"id": 3, "audio": "scene_03.mp3", "videos": ["Scene3aTree.mp4", "Scene3bMakeTasks.mp4"]},
    {"id": 4, "audio": "scene_04.mp3", "videos": ["Scene4aModelMacro.mp4", "Scene4bQuestionChoiceERD.mp4"]},
    {"id": 5, "audio": "scene_05.mp3", "videos": ["Scene5aServerFnRPC.mp4", "Scene5bUseActionCycle.mp4"]},
    {"id": 6, "audio": "scene_06.mp3", "videos": ["Scene6aFormMacro.mp4"]},
    {"id": 7, "audio": "scene_07.mp3", "videos": ["Scene7aAdminRegister.mp4"]},
    {"id": 8, "audio": "scene_08.mp3", "videos": ["Scene8aDIProviders.mp4", "Scene8bGuardChain.mp4"]},
    {"id": 9, "audio": "scene_09.mp3", "videos": ["Scene9Closing.mp4"]},
]

# Tail silence appended to each scene. Gives the listener a beat between
# scenes and is the easiest knob to stretch toward the 13:30 conte.md target.
TAIL_SILENCE = 0.6  # seconds

# Audio gain boost. moviepy + ffmpeg's mono→stereo AAC encode drops the
# RMS by ~3 dB; 1.5× ≈ +3.5 dB compensates without clipping.
AUDIO_GAIN = 1.5


In [ ]:
def build_video_for_scene(scene):
    """Concatenate this scene's Manim clips and freeze-pad to the audio length."""
    audio_dur = AudioFileClip(str(AUDIO_DIR / scene["audio"])).duration
    target = audio_dur + TAIL_SILENCE

    video_clips = [VideoFileClip(str(VIDEO_DIR / v)) for v in scene["videos"]]
    combined = (
        video_clips[0]
        if len(video_clips) == 1
        else concatenate_videoclips(video_clips, method="chain")
    )

    if combined.duration + 1e-3 < target:
        last_frame = combined.get_frame(max(0.0, combined.duration - 0.05))
        freeze = ImageClip(last_frame).with_duration(target - combined.duration)
        freeze.fps = combined.fps
        combined = concatenate_videoclips([combined, freeze], method="chain")
    elif combined.duration > target:
        combined = combined.subclipped(0, target)

    # Ensure the clip carries no audio track — audio is composed separately.
    return combined.with_duration(target).without_audio()


# Pre-flight duration plan.
print(f"{'SCENE':>5} {'AUDIO':>10} {'MANIM':>10} {'OUT':>10}")
total = 0.0
for scene in SCENES:
    a = AudioFileClip(str(AUDIO_DIR / scene["audio"])).duration
    v = sum(VideoFileClip(str(VIDEO_DIR / vn)).duration for vn in scene["videos"])
    out = a + TAIL_SILENCE
    total += out
    print(f"   {scene['id']:>2}  {a:8.2f}s  {v:8.2f}s  {out:8.2f}s")
print(f"TOTAL: {total:.2f}s  ({total / 60:.2f} min)")


In [ ]:
# 1. Build silent video for each scene, then concatenate.
scene_videos = [build_video_for_scene(s) for s in SCENES]
final_video = concatenate_videoclips(scene_videos, method="chain")

# 2. Build audio track: place each narration mp3 at the scene's cumulative start.
audio_segments = []
cursor = 0.0
for scene, sv in zip(SCENES, scene_videos, strict=True):
    a = AudioFileClip(str(AUDIO_DIR / scene["audio"])).with_start(cursor)
    audio_segments.append(a)
    cursor += sv.duration  # advance by the silent video's duration (incl. tail)

final_audio = CompositeAudioClip(audio_segments)
final_audio.fps = 44100  # required for AAC encoding
final_audio = final_audio.with_volume_scaled(AUDIO_GAIN).with_duration(cursor)

# 3. Attach audio.
final = final_video.with_audio(final_audio)

print(f"video duration: {final_video.duration:.2f}s")
print(f"audio duration: {final_audio.duration:.2f}s")
print(f"final  duration: {final.duration:.2f}s")
print(f"audio gain    : {AUDIO_GAIN}× (~{20 * np.log10(AUDIO_GAIN):.1f} dB)")


In [ ]:
print(f"Writing {OUT_PATH} ...")

final.write_videofile(
    str(OUT_PATH),
    fps=15,
    codec="libx264",
    audio_codec="aac",
    audio_bitrate="192k",
    threads=4,
    preset="medium",
)

print("Done.")


In [ ]:
# Sanity check the final mp4.
clip = VideoFileClip(str(OUT_PATH))
size_mb = OUT_PATH.stat().st_size / 1024 / 1024
print(f"path      : {OUT_PATH}")
print(f"size      : {size_mb:.1f} MB")
print(f"duration  : {clip.duration:.2f}s ({clip.duration / 60:.2f} min)")
print(f"fps       : {clip.fps}")
print(f"resolution: {clip.size}")
print(f"audio?    : {clip.audio is not None}")
if clip.audio is not None:
    print(f"audio fps : {clip.audio.fps}")
    print(f"audio dur : {clip.audio.duration:.2f}s")
clip.close()
